# Imaging Candidate Training — Segmentation + Detection

In [1]:
import sys
sys.path.insert(0, "/workspace")
from utils.config import (
    CACHE_DIR, CACHE_IMAGES_PATH, CACHE_MASKS_PATH, CACHE_MANIFEST_PATH, CACHE_META_PATH,
    CHECKPOINTS_IMAGING_CANDIDATES_DIR, RESULTS_IMAGING_DIR, IMAGING_OOF_PREDICTIONS_PATH, ensure_dirs,
)
from utils.metrics import dice_score, iou_score, detection_metrics

import json
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision.models import resnet50, ResNet50_Weights
from sklearn.metrics import roc_auc_score

ensure_dirs()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    print("WARNING: CUDA not available -- this notebook's timing/VRAM numbers assume the rented RTX 6000 Ada GPU path.")

# Fixed input shape (every slice is BOX x BOX from the cache) is exactly the precondition
# where cudnn's autotuner pays off -- measured in the timing diagnostic.
torch.backends.cudnn.benchmark = True

print(f"torch {torch.__version__}, cuda available: {torch.cuda.is_available()}, device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB VRAM")
print("Imports OK")

torch 2.8.0+cu128, cuda available: True, device: cuda
GPU: NVIDIA RTX 6000 Ada Generation, 47.50 GB VRAM
Imports OK


## Configuration

In [2]:
CANDIDATES = ["resnet50_unet", "baseline_cnn"]
FOLDS_TO_RUN = list(range(5))  # <-- 2026-07-17: extended from [0,1,2] to all 5 folds now that
# compute cost is trivial on this pod (~$0.35 for the 2 extra folds, ~27 extra min) -- the
# original 3-fold cap was a laptop-era schedule constraint, not a methodological choice, and
# the 3-fold OOF predictions only covered 60.7% of patients (folds 0-2). Full 5-fold gives
# every patient exactly one out-of-fold prediction. The prior 3-fold results are archived at
# checkpoints/imaging/candidates_3fold_archive/, results/imaging/archive/oof_predictions_3fold.csv,
# and docs/Imaging_3Fold_Training_Results_documentation.md before this rerun.

SLICE_STRIDE = 3  # train on every Nth training-fold slice per epoch, offset rotates
VAL_FIXED_STRIDE_OFFSET = 0  # fixed offset for the cheap per-epoch early-stopping subset

MAX_EPOCHS = 15
PATIENCE = 3  # epochs of no val-AUC improvement before stopping -- identical for both candidates

# CONFIRMED 2026-07-17 from the standalone pre-flight sweep on this pod (reported to and
# approved by the user before the first full run): batch=32 beat both 16 and 64 on throughput
# (201.7 img/s vs. 190.9 and 193.0), and torch.compile on top of batch=32 measurably helped
# further (+6.2%, 214.2 img/s) with no VRAM concern (peak 5.6-22.5GB, well under the 48GB
# card). These are no longer provisional -- the Pre-Flight section further down re-validates
# on this specific run but does not gate these values.
BATCH_SIZE = 32
NUM_WORKERS = 4
LEARNING_RATE = 1e-4
USE_TORCH_COMPILE = True  # applied to resnet50_unet only (the candidate it was validated on) -- see CANDIDATE_SPECS

RUN_FULL_TRAINING = True  # <-- confirmed by the user after reviewing the pre-flight timing/cost estimate

print(f"Candidates: {CANDIDATES}")
print(f"Folds to run: {FOLDS_TO_RUN}")
print(f"Slice stride: {SLICE_STRIDE}, batch size: {BATCH_SIZE}, num_workers: {NUM_WORKERS}")
print(f"Max epochs: {MAX_EPOCHS}, patience: {PATIENCE}")
print(f"torch.compile: {USE_TORCH_COMPILE}")
print(f"RUN_FULL_TRAINING: {RUN_FULL_TRAINING}")

Candidates: ['resnet50_unet', 'baseline_cnn']
Folds to run: [0, 1, 2, 3, 4]
Slice stride: 3, batch size: 32, num_workers: 4
Max epochs: 15, patience: 3
torch.compile: True
RUN_FULL_TRAINING: True


## Load the Packed Cache

In [3]:
with open(CACHE_META_PATH) as f:
    cache_meta = json.load(f)
print("cache_meta.json:", json.dumps(cache_meta, indent=2))

BOX_SIZE = cache_meta["box_size"]
assert BOX_SIZE == 320, f"Expected BOX=320 (repacked after moving to the rented GPU), got {BOX_SIZE}"

manifest_df = pd.read_csv(CACHE_MANIFEST_PATH)
print(f"\nmanifest_cache: {len(manifest_df):,} rows")
print(manifest_df["split"].value_counts().sort_index())
print(manifest_df.drop_duplicates("patient_id").groupby(["dataset", "class"]).size())

assert manifest_df["img_row"].nunique() == len(manifest_df), "img_row must be unique per row"
n_msd = int((manifest_df["dataset"] == "MSD").sum())
assert (manifest_df.loc[manifest_df["dataset"] == "MSD", "mask_row"] >= 0).all(), "every MSD row must have a valid mask_row"
assert (manifest_df.loc[manifest_df["dataset"] == "NIH", "mask_row"] == -1).all(), "every NIH row must have mask_row == -1"
print(f"\nRow-index integrity OK: {len(manifest_df)} img_rows, {n_msd} MSD mask_rows, NIH rows all mask_row=-1")

cache_meta.json: {
  "box_size": 320,
  "dtype": "uint8",
  "n_images": 90693,
  "n_masks": 72077,
  "pad_value_image": 0,
  "pad_value_mask": 0,
  "source_manifest_path": "C:\\FYP\\data\\processed\\manifest.csv",
  "source_manifest_sha256": "d6209aae0ec5fe9580672ce7543546bac2b8289f7bf91455c8acf1cf640d1f59",
  "source_manifest_mtime": "2026-07-14T15:47:23.775770",
  "created_at": "2026-07-17T10:56:51.867365"
}

manifest_cache: 90,693 rows
split
fold0    18554
fold1    18763
fold2    17739
fold3    17398
fold4    18239
Name: count, dtype: int64
dataset  class
MSD      1        281
NIH      0         80
dtype: int64



Row-index integrity OK: 90693 img_rows, 72077 MSD mask_rows, NIH rows all mask_row=-1


## Cache-Backed Dataset

In [4]:
from imaging.slice_cache_dataset import SliceCacheDataset

print("SliceCacheDataset imported from imaging.slice_cache_dataset")

SliceCacheDataset imported from imaging.slice_cache_dataset


## Fold Splitting, Strided Sampling, Fixed Validation Subset

In [5]:
def get_fold_split(manifest_df: pd.DataFrame, fold: int) -> tuple:
    """Train = every row NOT labeled fold{fold}; val = rows labeled fold{fold}. Reads the
    manifest's existing split column as-is -- no fold is re-derived here."""
    train_df = manifest_df[manifest_df["split"] != f"fold{fold}"].reset_index(drop=True)
    val_df = manifest_df[manifest_df["split"] == f"fold{fold}"].reset_index(drop=True)
    return train_df, val_df


def strided_pool(df: pd.DataFrame, stride: int, epoch: int) -> pd.DataFrame:
    """Filter to slice_index % stride == epoch % stride. TRAINING FOLDS ONLY -- lossless
    across `stride` consecutive epochs since every slice is included in exactly one of
    them, just with the epoch boundary moved."""
    offset = epoch % stride
    return df[df["slice_index"] % stride == offset].reset_index(drop=True)


def build_weighted_sampler(df: pd.DataFrame) -> WeightedRandomSampler:
    """WeightedRandomSampler over df['class'], built on the ALREADY-STRIDED pool -- must be
    called after strided_pool, never before, or the class rebalancing this exists for gets
    silently undone by the subsequent striding."""
    class_counts = df["class"].value_counts()
    class_weight = 1.0 / class_counts
    sample_weights = df["class"].map(class_weight).to_numpy()
    return WeightedRandomSampler(weights=sample_weights, num_samples=len(df), replacement=True)


def build_fixed_val_subset(val_df: pd.DataFrame, stride: int, offset: int = VAL_FIXED_STRIDE_OFFSET) -> pd.DataFrame:
    """Fixed (never-rotating) strided subset of the val fold -- same rows every epoch, used
    only for the cheap per-epoch early-stopping AUC signal, never for reported metrics."""
    return val_df[val_df["slice_index"] % stride == offset].reset_index(drop=True)


print("Fold/sampling helpers defined")

Fold/sampling helpers defined


## Models

In [6]:
class UpBlock(nn.Module):
    """One U-Net decoder stage: upsample 2x, concat with the matching encoder skip
    connection, then two conv/BN/ReLU layers to mix the concatenated features."""

    def __init__(self, in_ch: int, skip_ch: int, out_ch: int):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(out_ch + skip_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class ResNet50UNet(nn.Module):
    """Pretrained ResNet-50 encoder + U-Net decoder (segmentation) + a detection head off
    the encoder bottleneck. 3-channel input (replicated from 1 by the Dataset, not stored
    that way). num_seg_classes=3: background/pancreas/tumour."""

    def __init__(self, num_seg_classes: int = 3):
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)  # /2, 64ch
        self.maxpool = backbone.maxpool  # /4 total
        self.layer1 = backbone.layer1  # /4, 256ch
        self.layer2 = backbone.layer2  # /8, 512ch
        self.layer3 = backbone.layer3  # /16, 1024ch
        self.layer4 = backbone.layer4  # /32, 2048ch

        self.up4 = UpBlock(2048, 1024, 1024)  # -> /16, matches layer3
        self.up3 = UpBlock(1024, 512, 512)  # -> /8, matches layer2
        self.up2 = UpBlock(512, 256, 256)  # -> /4, matches layer1
        self.up1 = UpBlock(256, 64, 64)  # -> /2, matches stem output
        self.final_up = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)  # -> /1, full res
        self.seg_head = nn.Conv2d(32, num_seg_classes, kernel_size=1)

        self.det_pool = nn.AdaptiveAvgPool2d(1)
        self.det_head = nn.Linear(2048, 1)

    def forward(self, x: torch.Tensor) -> tuple:
        s0 = self.stem(x)
        p0 = self.maxpool(s0)
        l1 = self.layer1(p0)
        l2 = self.layer2(l1)
        l3 = self.layer3(l2)
        l4 = self.layer4(l3)

        d4 = self.up4(l4, l3)
        d3 = self.up3(d4, l2)
        d2 = self.up2(d3, l1)
        d1 = self.up1(d2, s0)
        d0 = self.final_up(d1)
        seg_logits = self.seg_head(d0)

        det_feat = self.det_pool(l4).flatten(1)
        det_logit = self.det_head(det_feat).squeeze(-1)

        return seg_logits, det_logit


class BaselineCNN(nn.Module):
    """From-scratch detection-only CNN, single-channel input. Same architecture measured
    in the timing diagnostic: 4x [conv/BN/ReLU/MaxPool] (16->32->64->128 channels), GAP,
    one logit."""

    def __init__(self):
        super().__init__()

        def block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )

        self.features = nn.Sequential(block(1, 16), block(16, 32), block(32, 64), block(64, 128))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Linear(128, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        f = self.features(x)
        f = self.pool(f).flatten(1)
        return self.head(f).squeeze(-1)


CANDIDATE_SPECS = {
    # use_compile=True only for resnet50_unet -- the candidate torch.compile was actually
    # measured on in the pre-flight sweep (+6.2% at batch=32). baseline_cnn is untested with
    # compile and is already fast/tiny, so it stays eager rather than assuming the benefit
    # transfers.
    # augment=True only for resnet50_unet, added 2026-07-17 -- the confound check found this
    # candidate's detection head relies significantly MORE on synthetic padding than a random
    # patch of tissue and significantly LESS on the real tumour region (see markdown above).
    # baseline_cnn is untouched: not the winning candidate, not the subject of that check.
    # Retrain 2026-07-17 (Round 6): SliceCacheDataset's augment=True path now also applies
    # mask-preserving random erase (MSD rows only) -- targets the remaining tumour
    # under-reliance found by the Round 5 occlusion re-test. No new flag needed here: it
    # rides the same augment=True switch. See imaging/slice_cache_dataset.py.
    "resnet50_unet": {"builder": lambda: ResNet50UNet(), "use_3channel": True, "has_seg_head": True,
                       "use_compile": True, "augment": True},
    "baseline_cnn": {"builder": lambda: BaselineCNN(), "use_3channel": False, "has_seg_head": False,
                      "use_compile": False, "augment": False},
}

print("Models defined:", list(CANDIDATE_SPECS.keys()))

Models defined: ['resnet50_unet', 'baseline_cnn']


## Losses

In [7]:
detection_loss_fn = nn.BCEWithLogitsLoss()


def dice_loss_from_logits(seg_logits: torch.Tensor, mask: torch.Tensor, num_classes: int = 3, eps: float = 1e-6) -> torch.Tensor:
    """Differentiable soft Dice loss (1 - Dice), averaged over foreground classes
    (1..num_classes-1), from raw segmentation logits and an integer mask (N,H,W). The
    training-time counterpart to utils.metrics.dice_score (same foreground-only, same
    class convention), but soft/differentiable instead of hard-thresholded."""
    probs = F.softmax(seg_logits, dim=1)
    mask_onehot = F.one_hot(mask, num_classes=num_classes).permute(0, 3, 1, 2).float()
    losses = []
    for c in range(1, num_classes):
        p, t = probs[:, c], mask_onehot[:, c]
        intersection = (p * t).sum(dim=(1, 2))
        denom = p.sum(dim=(1, 2)) + t.sum(dim=(1, 2))
        dice = (2 * intersection + eps) / (denom + eps)
        losses.append(1 - dice)
    return torch.stack(losses).mean()


def segmentation_loss(seg_logits: torch.Tensor, mask: torch.Tensor, num_classes: int = 3) -> torch.Tensor:
    """Combined Dice + CrossEntropy segmentation loss, MSD-only rows already filtered out
    by the caller before this is invoked."""
    ce = F.cross_entropy(seg_logits, mask)
    dice = dice_loss_from_logits(seg_logits, mask, num_classes)
    return ce + dice


def compute_batch_loss(model, images, labels, masks, has_mask, has_seg_head: bool) -> dict:
    """Runs the model forward and returns det_loss, seg_loss, and their sum -- seg_loss is
    a harmless 0.0 tensor when has_seg_head is False (baseline CNN) or when no row in this
    batch has a real mask (all-NIH batch)."""
    if has_seg_head:
        seg_logits, det_logit = model(images)
        det_loss = detection_loss_fn(det_logit, labels)
        if has_mask.any():
            seg_loss = segmentation_loss(seg_logits[has_mask], masks[has_mask])
        else:
            seg_loss = torch.zeros((), device=images.device)
    else:
        det_logit = model(images)
        det_loss = detection_loss_fn(det_logit, labels)
        seg_loss = torch.zeros((), device=images.device)

    return {"det_logit": det_logit, "det_loss": det_loss, "seg_loss": seg_loss, "loss": det_loss + seg_loss}


print("Loss functions defined")

Loss functions defined


## Metrics

## Training Loop

In [8]:
def train_one_epoch(model, optimizer, scaler, train_df, epoch, use_3channel, has_seg_head,
                     batch_size=None, augment=False) -> dict:
    """One training epoch: rebuild the strided pool + weighted sampler for THIS epoch
    (required every epoch since the stride offset rotates), then train with AMP +
    channels_last. Returns average losses, batch count, and realised class balance.

    batch_size defaults to the global BATCH_SIZE -- overridable so the pre-flight sweep
    can reuse this exact loop at batch=16/32/64 without duplicating it.

    augment=True (added 2026-07-17) applies SliceCacheDataset's random-resized-crop at
    train time only -- see CANDIDATE_SPECS and the Models section markdown for why."""
    batch_size = batch_size or BATCH_SIZE
    pool = strided_pool(train_df, SLICE_STRIDE, epoch)
    sampler = build_weighted_sampler(pool)
    dataset = SliceCacheDataset(pool, CACHE_IMAGES_PATH, CACHE_MASKS_PATH, use_3channel, BOX_SIZE, augment=augment)
    loader = DataLoader(dataset, batch_size=batch_size, sampler=sampler, num_workers=NUM_WORKERS,
                         pin_memory=True, drop_last=True)

    model.train()
    total_det_loss, total_seg_loss, n_batches = 0.0, 0.0, 0
    class_counts = {0: 0, 1: 0}

    for batch in loader:
        images = batch["image"].to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
        labels = batch["label"].to(DEVICE, non_blocking=True)
        masks = batch["mask"].to(DEVICE, non_blocking=True)
        has_mask = batch["has_mask"].to(DEVICE, non_blocking=True)

        for cls_val, cnt in zip(*torch.unique(batch["label"], return_counts=True)):
            class_counts[int(cls_val.item())] = class_counts.get(int(cls_val.item()), 0) + int(cnt.item())

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
            out = compute_batch_loss(model, images, labels, masks, has_mask, has_seg_head)

        scaler.scale(out["loss"]).backward()
        scaler.step(optimizer)
        scaler.update()

        total_det_loss += out["det_loss"].item()
        total_seg_loss += float(out["seg_loss"].item()) if torch.is_tensor(out["seg_loss"]) else float(out["seg_loss"])
        n_batches += 1

    return {
        "avg_det_loss": total_det_loss / max(n_batches, 1),
        "avg_seg_loss": total_seg_loss / max(n_batches, 1),
        "n_batches": n_batches,
        "n_rows": len(pool),
        "batch_size": batch_size,
        "class_counts": class_counts,
    }


def evaluate_fixed_subset(model, fixed_val_df, use_3channel, has_seg_head) -> float:
    """Inference-only detection AUC over the FIXED strided val subset -- cheap per-epoch
    early-stopping signal only, never reported as a final metric. augment=False always
    (the default) -- validation must reflect real, unaugmented images."""
    dataset = SliceCacheDataset(fixed_val_df, CACHE_IMAGES_PATH, CACHE_MASKS_PATH, use_3channel, BOX_SIZE)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
                det_logit = model(images)[1] if has_seg_head else model(images)
            all_probs.append(torch.sigmoid(det_logit).float().cpu())
            all_labels.append(batch["label"])

    all_probs = torch.cat(all_probs).numpy()
    all_labels = torch.cat(all_labels).numpy()
    if len(np.unique(all_labels)) <= 1:
        return float("nan")
    return float(roc_auc_score(all_labels, all_probs))


def evaluate_full(model, val_df, use_3channel, has_seg_head, candidate_name, fold) -> dict:
    """FULL, UNSTRIDED pass over the entire val fold -- every reported metric comes from
    here, never from the fixed strided subset used for early stopping. augment=False always
    (the default) -- reported metrics and OOF predictions must reflect real, unaugmented
    images."""
    dataset = SliceCacheDataset(val_df, CACHE_IMAGES_PATH, CACHE_MASKS_PATH, use_3channel, BOX_SIZE)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    model.eval()
    all_probs, all_labels = [], []
    all_seg_pred, all_seg_true = [], []
    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
            has_mask = batch["has_mask"]
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
                if has_seg_head:
                    seg_logits, det_logit = model(images)
                else:
                    det_logit = model(images)

            all_probs.append(torch.sigmoid(det_logit).float().cpu())
            all_labels.append(batch["label"])

            if has_seg_head and has_mask.any():
                mask_np = has_mask.numpy()
                all_seg_pred.append(seg_logits.argmax(dim=1).cpu().numpy()[mask_np])
                all_seg_true.append(batch["mask"].numpy()[mask_np])

    all_probs = torch.cat(all_probs).numpy()
    all_labels = torch.cat(all_labels).numpy()
    y_pred = (all_probs >= 0.5).astype(int)

    det = detection_metrics(all_labels, y_pred, all_probs)

    seg_metrics = None
    if has_seg_head and all_seg_pred:
        seg_metrics = {
            "dice": dice_score(np.concatenate(all_seg_pred), np.concatenate(all_seg_true)),
            "iou": iou_score(np.concatenate(all_seg_pred), np.concatenate(all_seg_true)),
        }

    oof_rows = pd.DataFrame({
        "candidate": candidate_name, "fold": fold,
        "patient_id": val_df["patient_id"].values, "slice_index": val_df["slice_index"].values,
        "dataset": val_df["dataset"].values,
        "y_true": all_labels, "y_proba": all_probs, "y_pred": y_pred,
    })

    return {"detection": det, "segmentation": seg_metrics, "oof_rows": oof_rows}


print("Training/eval functions defined")

Training/eval functions defined


## Per-Candidate, Per-Fold Orchestration

In [9]:
def unwrap_model(model: nn.Module) -> nn.Module:
    """Returns the underlying module when `model` is a torch.compile() OptimizedModule
    (its state_dict() keys are prefixed '_orig_mod.', which a plain, uncompiled model
    instantiated later -- e.g. in imaging_evaluation.ipynb -- cannot load directly). Always
    save/restore checkpoints through this so they stay compile-agnostic on disk. Confirmed
    empirically: OptimizedModule.state_dict() keeps the '_orig_mod.' prefix on torch 2.8."""
    return model._orig_mod if hasattr(model, "_orig_mod") else model


def run_training_for_fold(candidate_name: str, fold: int, manifest_df: pd.DataFrame,
                           max_epochs: int = MAX_EPOCHS, patience: int = PATIENCE, verbose: bool = True) -> dict:
    """Train one candidate on one fold: early stopping on fixed-subset val AUC (patience,
    ceiling identical for both candidates), restore best-epoch weights, run the full
    unstrided val evaluation, save the checkpoint, return history + metrics + OOF rows."""
    spec = CANDIDATE_SPECS[candidate_name]
    train_df, val_df = get_fold_split(manifest_df, fold)
    fixed_val_subset = build_fixed_val_subset(val_df, SLICE_STRIDE, VAL_FIXED_STRIDE_OFFSET)

    model = spec["builder"]().to(DEVICE, memory_format=torch.channels_last)
    n_params = sum(p.numel() for p in model.parameters())
    if spec.get("use_compile") and USE_TORCH_COMPILE:
        model = torch.compile(model)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

    best_auc, best_epoch, best_state, epochs_since_improve = -float("inf"), None, None, 0
    history = []

    for epoch in range(max_epochs):
        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()

        train_stats = train_one_epoch(model, optimizer, scaler, train_df, epoch, spec["use_3channel"], spec["has_seg_head"],
                                       augment=spec.get("augment", False))
        val_auc = evaluate_fixed_subset(model, fixed_val_subset, spec["use_3channel"], spec["has_seg_head"])

        elapsed = time.perf_counter() - t0
        peak_vram_gb = torch.cuda.max_memory_allocated() / 1024**3 if DEVICE.type == "cuda" else float("nan")

        history.append({
            "epoch": epoch, "train_det_loss": train_stats["avg_det_loss"], "train_seg_loss": train_stats["avg_seg_loss"],
            "val_auc": val_auc, "class_counts": train_stats["class_counts"],
            "wall_clock_sec": elapsed, "peak_vram_gb": peak_vram_gb,
        })
        if verbose:
            print(f"[{candidate_name} fold{fold}] epoch {epoch}: det_loss={train_stats['avg_det_loss']:.4f} "
                  f"seg_loss={train_stats['avg_seg_loss']:.4f} val_auc={val_auc:.4f} "
                  f"class_balance={train_stats['class_counts']} time={elapsed:.1f}s peak_vram={peak_vram_gb:.3f}GB")

        improved = (not np.isnan(val_auc)) and (val_auc > best_auc)
        if improved:
            best_auc, best_epoch = val_auc, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in unwrap_model(model).state_dict().items()}
            epochs_since_improve = 0
        else:
            epochs_since_improve += 1

        if epochs_since_improve >= patience:
            if verbose:
                print(f"[{candidate_name} fold{fold}] early stop at epoch {epoch} "
                      f"(best epoch {best_epoch}, best val_auc {best_auc:.4f})")
            break

    stopped_epoch = history[-1]["epoch"]
    if best_state is not None:
        unwrap_model(model).load_state_dict(best_state)

    full_results = evaluate_full(model, val_df, spec["use_3channel"], spec["has_seg_head"], candidate_name, fold)

    candidate_dir = CHECKPOINTS_IMAGING_CANDIDATES_DIR / candidate_name
    candidate_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = candidate_dir / f"fold_{fold}.pt"
    torch.save(unwrap_model(model).state_dict(), checkpoint_path)

    return {
        "candidate": candidate_name, "fold": fold, "n_params": n_params, "history": history,
        "stopped_epoch": stopped_epoch, "best_epoch": best_epoch, "best_val_auc": best_auc,
        "full_results": full_results, "checkpoint_path": str(checkpoint_path),
    }


print("run_training_for_fold defined")

run_training_for_fold defined


## Full Run (gated by `RUN_FULL_TRAINING`)

In [10]:
if not RUN_FULL_TRAINING:
    print("RUN_FULL_TRAINING is False -- skipping the full run.")
    print("Flip the flag in the Configuration cell after reviewing the pre-flight timing below.")
else:
    all_results = {}
    oof_frames = []

    for candidate_name in CANDIDATES:
        candidate_dir = CHECKPOINTS_IMAGING_CANDIDATES_DIR / candidate_name
        candidate_dir.mkdir(parents=True, exist_ok=True)
        fold_results = []

        for fold in FOLDS_TO_RUN:
            result = run_training_for_fold(candidate_name, fold, manifest_df)
            fold_results.append(result)
            oof_frames.append(result["full_results"]["oof_rows"])

        all_results[candidate_name] = fold_results

        model_card = {
            "candidate": candidate_name,
            "folds_run": list(FOLDS_TO_RUN),
            "n_params": fold_results[0]["n_params"],
            "max_epochs": MAX_EPOCHS,
            "patience": PATIENCE,
            "slice_stride": SLICE_STRIDE,
            "batch_size": BATCH_SIZE,
            "per_fold": [
                {
                    "fold": r["fold"],
                    "stopped_epoch": r["stopped_epoch"],
                    "best_epoch": r["best_epoch"],
                    "best_val_auc": r["best_val_auc"],
                    "detection_metrics": {
                        k: (v.tolist() if isinstance(v, np.ndarray) else v)
                        for k, v in r["full_results"]["detection"].items()
                    },
                    "segmentation_metrics": r["full_results"]["segmentation"],
                }
                for r in fold_results
            ],
        }
        with open(candidate_dir / "model_card.json", "w") as f:
            json.dump(model_card, f, indent=2, default=str)
        print(f"Wrote {candidate_dir / 'model_card.json'}")

    oof_df = pd.concat(oof_frames, ignore_index=True)
    oof_df.to_csv(IMAGING_OOF_PREDICTIONS_PATH, index=False)
    print(f"Wrote {IMAGING_OOF_PREDICTIONS_PATH} ({len(oof_df)} rows)")

    print("\n=== Comparison Summary (mean across folds actually run) ===")
    summary_rows = []
    for candidate_name, fold_results in all_results.items():
        summary_rows.append({
            "candidate": candidate_name,
            "mean_roc_auc": np.nanmean([r["full_results"]["detection"]["roc_auc"] for r in fold_results]),
            "mean_recall": np.nanmean([r["full_results"]["detection"]["recall"] for r in fold_results]),
            "mean_precision": np.nanmean([r["full_results"]["detection"]["precision"] for r in fold_results]),
            "mean_stopped_epoch": np.mean([r["stopped_epoch"] for r in fold_results]),
        })
    summary_df = pd.DataFrame(summary_rows)
    print(summary_df.to_string(index=False))

    if len(summary_df) == 2:
        auc_gap = abs(summary_df.iloc[0]["mean_roc_auc"] - summary_df.iloc[1]["mean_roc_auc"])
        winner = summary_df.loc[summary_df["mean_roc_auc"].idxmax(), "candidate"]
        if auc_gap < 0.02:
            print(f"\nROC-AUC gap ({auc_gap:.4f}) is small -- no clear winner on detection ROC-AUC alone. "
                  f"Check recall (clinically the more important number) and segmentation Dice/IoU above "
                  f"before declaring a winner.")
        else:
            print(f"\nWinner (detection ROC-AUC): {winner} (gap {auc_gap:.4f})")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


  0%|          | 0.00/97.8M [00:00<?, ?B/s]

 38%|███▊      | 37.6M/97.8M [00:00<00:00, 394MB/s]

 77%|███████▋  | 75.8M/97.8M [00:00<00:00, 397MB/s]

100%|██████████| 97.8M/97.8M [00:00<00:00, 385MB/s]

...

[baseline_cnn fold4] epoch 3: det_loss=0.2111 seg_loss=0.0000 val_auc=0.9708 class_balance={0: 12121, 1: 12103} time=27.3s peak_vram=0.959GB


[baseline_cnn fold4] epoch 4: det_loss=0.1654 seg_loss=0.0000 val_auc=0.9646 class_balance={0: 12063, 1: 12065} time=25.8s peak_vram=0.959GB


[baseline_cnn fold4] epoch 5: det_loss=0.1452 seg_loss=0.0000 val_auc=0.9688 class_balance={0: 11983, 1: 12049} time=26.1s peak_vram=0.959GB
[baseline_cnn fold4] early stop at epoch 5 (best epoch 2, best val_auc 0.9715)


Wrote /workspace/checkpoints/imaging/candidates/baseline_cnn/model_card.json


Wrote /workspace/results/imaging/oof_predictions.csv (181386 rows)

=== Comparison Summary (mean across folds actually run) ===
    candidate  mean_roc_auc  mean_recall  mean_precision  mean_stopped_epoch
resnet50_unet      0.992786     0.979043        0.983222                 5.4
 baseline_cnn      0.978933     0.840713        0.952767                10.4

ROC-AUC gap (0.0139) is small -- no clear winner on detection ROC-AUC alone. Check recall (clinically the more important number) and segmentation Dice/IoU above before declaring a winner.


## Pre-Flight: Batch-Size Sweep + torch.compile Attempt, ResNet-50 U-Net Only

In [11]:
PREFLIGHT_FOLD = FOLDS_TO_RUN[0]
PREFLIGHT_CANDIDATE = "resnet50_unet"
LOCAL_BASELINE_MIN_PER_EPOCH = 37.81  # RTX 3050, BOX=256, batch=16, confirmed compute-bound (97% GPU util)

print(f"Pre-flight: {PREFLIGHT_CANDIDATE}, fold {PREFLIGHT_FOLD}")

spec = CANDIDATE_SPECS[PREFLIGHT_CANDIDATE]
train_df, val_df = get_fold_split(manifest_df, PREFLIGHT_FOLD)
print(f"Train rows (fold {PREFLIGHT_FOLD} held out): {len(train_df):,}")

preflight_results = {}  # keyed by label, e.g. "batch16", "batch32", "batch64", "batch32_compiled"


def run_preflight_epoch(label: str, batch_size: int, compiled: bool = False) -> dict:
    """Fresh model + optimizer, one real training epoch (epoch=0, stride offset 0, same
    fold every call), measuring wall-clock, img/s, and peak VRAM. Stores into
    preflight_results[label] and returns the same dict. Uses spec["augment"] so this
    pre-flight timing reflects the actual training config (augmentation is a cheap CPU-side
    op but threaded through for consistency, not assumed negligible without checking)."""
    model = spec["builder"]().to(DEVICE, memory_format=torch.channels_last)
    if compiled:
        model = torch.compile(model)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()

    stats = train_one_epoch(model, optimizer, scaler, train_df, epoch=0,
                             use_3channel=spec["use_3channel"], has_seg_head=spec["has_seg_head"],
                             batch_size=batch_size, augment=spec.get("augment", False))

    elapsed_sec = time.perf_counter() - t0
    peak_vram_gb = torch.cuda.max_memory_allocated() / 1024**3 if DEVICE.type == "cuda" else float("nan")
    img_per_sec = stats["n_rows"] / elapsed_sec

    result = {
        "label": label, "batch_size": batch_size, "compiled": compiled,
        "n_rows": stats["n_rows"], "n_batches": stats["n_batches"],
        "elapsed_sec": elapsed_sec, "elapsed_min": elapsed_sec / 60,
        "img_per_sec": img_per_sec, "peak_vram_gb": peak_vram_gb,
        "class_counts": stats["class_counts"],
    }
    preflight_results[label] = result

    print(f"\n=== {label}: batch={batch_size}, compiled={compiled} ===")
    print(f"Rows: {stats['n_rows']:,} ({stats['n_batches']} batches)")
    print(f"Wall-clock: {elapsed_sec:.1f}s ({elapsed_sec / 60:.2f} min)")
    print(f"Throughput: {img_per_sec:.1f} img/s")
    print(f"Peak VRAM: {peak_vram_gb:.3f} GB / 48 GB available")
    print(f"Realised class balance: {stats['class_counts']}")

    del model, optimizer, scaler
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return result


# Step 1: batch=16, same basis as the local RTX 3050 measurement
run_preflight_epoch("batch16", batch_size=16)

pod_min = preflight_results["batch16"]["elapsed_min"]
speedup = LOCAL_BASELINE_MIN_PER_EPOCH / pod_min
print(f"\n=== vs. local baseline ===")
print(f"Local (RTX 3050, BOX=256, batch=16): {LOCAL_BASELINE_MIN_PER_EPOCH:.2f} min/epoch")
print(f"Pod   (RTX 6000 Ada, BOX=320, batch=16): {pod_min:.2f} min/epoch")
print(f"Speedup: {speedup:.2f}x (pod is also at BOX=320 vs local's BOX=256 -- more pixels per "
      f"image -- so this understates the pure hardware speedup somewhat)")

Pre-flight: resnet50_unet, fold 0
Train rows (fold 0 held out): 72,139



=== batch16: batch=16, compiled=False ===
Rows: 24,140 (1508 batches)
Wall-clock: 127.9s (2.13 min)
Throughput: 188.7 img/s
Peak VRAM: 19.856 GB / 48 GB available
Realised class balance: {0: 12210, 1: 11918}

=== vs. local baseline ===
Local (RTX 3050, BOX=256, batch=16): 37.81 min/epoch
Pod   (RTX 6000 Ada, BOX=320, batch=16): 2.13 min/epoch
Speedup: 17.73x (pod is also at BOX=320 vs local's BOX=256 -- more pixels per image -- so this understates the pure hardware speedup somewhat)


In [12]:
# Step 2: batch=32 and batch=64 -- same fold, same stride offset, fresh model each time
run_preflight_epoch("batch32", batch_size=32)
run_preflight_epoch("batch64", batch_size=64)

print("\n=== Batch-size sweep summary (eager mode) ===")
for label in ["batch16", "batch32", "batch64"]:
    r = preflight_results[label]
    print(f"  {label}: {r['img_per_sec']:.1f} img/s, {r['elapsed_min']:.2f} min/epoch, peak VRAM {r['peak_vram_gb']:.2f} GB")

best_eager_label = max(["batch16", "batch32", "batch64"], key=lambda l: preflight_results[l]["img_per_sec"])
best_eager_batch = preflight_results[best_eager_label]["batch_size"]
print(f"\nBest eager-mode throughput: {best_eager_label} ({preflight_results[best_eager_label]['img_per_sec']:.1f} img/s)")


=== batch32: batch=32, compiled=False ===
Rows: 24,140 (754 batches)
Wall-clock: 82.6s (1.38 min)
Throughput: 292.1 img/s
Peak VRAM: 6.045 GB / 48 GB available
Realised class balance: {0: 12053, 1: 12075}



=== batch64: batch=64, compiled=False ===
Rows: 24,140 (377 batches)
Wall-clock: 130.5s (2.18 min)
Throughput: 184.9 img/s
Peak VRAM: 28.531 GB / 48 GB available
Realised class balance: {0: 12178, 1: 11950}



=== Batch-size sweep summary (eager mode) ===
  batch16: 188.7 img/s, 2.13 min/epoch, peak VRAM 19.86 GB
  batch32: 292.1 img/s, 1.38 min/epoch, peak VRAM 6.04 GB
  batch64: 184.9 img/s, 2.18 min/epoch, peak VRAM 28.53 GB

Best eager-mode throughput: batch32 (292.1 img/s)


In [13]:
# Step 3: torch.compile attempt, at the batch size that won the eager sweep
print(f"Attempting torch.compile() at {best_eager_label} (batch={best_eager_batch})...")
try:
    run_preflight_epoch(f"{best_eager_label}_compiled", batch_size=best_eager_batch, compiled=True)
    compile_ok = True
except Exception as e:
    compile_ok = False
    print(f"\n*** torch.compile() FAILED: {type(e).__name__}: {e} ***")
    print("Dropping torch.compile for the real run -- not silently retried or fought.")

if compile_ok:
    eager_img_s = preflight_results[best_eager_label]["img_per_sec"]
    compiled_img_s = preflight_results[f"{best_eager_label}_compiled"]["img_per_sec"]
    compile_speedup = compiled_img_s / eager_img_s
    print(f"\n=== torch.compile comparison at batch={best_eager_batch} ===")
    print(f"Eager:    {eager_img_s:.1f} img/s")
    print(f"Compiled: {compiled_img_s:.1f} img/s")
    print(f"Compile speedup: {compile_speedup:.3f}x")
    print("Note: this epoch includes torch.compile's first-call tracing/autotuning overhead "
          "(no separate warmup pass) -- a real multi-epoch run amortizes that one-time cost "
          "further, so this understates compile's steady-state benefit if it helps at all.")
    if compile_speedup > 1.05:
        print(f"torch.compile measurably helps (+{(compile_speedup - 1) * 100:.1f}%) -- recommend using it for the real run.")
        PREFLIGHT_RECOMMENDED_COMPILE = True
    else:
        print(f"torch.compile does not measurably help ({(compile_speedup - 1) * 100:+.1f}%) -- dropping it for the real run.")
        PREFLIGHT_RECOMMENDED_COMPILE = False
else:
    PREFLIGHT_RECOMMENDED_COMPILE = False

RECOMMENDED_BATCH_SIZE = best_eager_batch
print(f"\nRecommended batch size: {RECOMMENDED_BATCH_SIZE}")
print(f"Recommend torch.compile: {PREFLIGHT_RECOMMENDED_COMPILE}")
print("\nNOTE: this is a re-validation of the sweep already reported to and confirmed by the "
      "user (BATCH_SIZE=32, USE_TORCH_COMPILE=True in the Configuration cell above, which the "
      "Full Run section already used earlier in this same execution). This section intentionally "
      "uses its own PREFLIGHT_RECOMMENDED_* names rather than overwriting BATCH_SIZE/"
      "USE_TORCH_COMPILE, so it never silently changes an already-confirmed config.")

Attempting torch.compile() at batch32 (batch=32)...



=== batch32_compiled: batch=32, compiled=True ===
Rows: 24,140 (754 batches)
Wall-clock: 81.0s (1.35 min)
Throughput: 297.9 img/s
Peak VRAM: 5.949 GB / 48 GB available
Realised class balance: {0: 11947, 1: 12181}

=== torch.compile comparison at batch=32 ===
Eager:    292.1 img/s
Compiled: 297.9 img/s
Compile speedup: 1.020x
Note: this epoch includes torch.compile's first-call tracing/autotuning overhead (no separate warmup pass) -- a real multi-epoch run amortizes that one-time cost further, so this understates compile's steady-state benefit if it helps at all.
torch.compile does not measurably help (+2.0%) -- dropping it for the real run.

Recommended batch size: 32
Recommend torch.compile: False

NOTE: this is a re-validation of the sweep already reported to and confirmed by the user (BATCH_SIZE=32, USE_TORCH_COMPILE=True in the Configuration cell above, which the Full Run section already used earlier in this same execution). This section intentionally uses its own PREFLIGHT_RECOMM

In [14]:
# Step 4: Extrapolation to the full run (3 folds x 2 candidates) at the recommended config
best_label = f"{best_eager_label}_compiled" if PREFLIGHT_RECOMMENDED_COMPILE else best_eager_label
chosen = preflight_results[best_label]
minutes_per_epoch = chosen["elapsed_min"]
n_folds = len(FOLDS_TO_RUN)
n_candidates = len(CANDIDATES)

print(f"=== Extrapolation using {best_label} ({minutes_per_epoch:.2f} min/epoch) ===")
print("NOTE: this rate was measured on resnet50_unet only. baseline_cnn is a much smaller, "
      "from-scratch model (no decoder, single-channel input) and is expected to be "
      "substantially faster per epoch -- applying the resnet50_unet rate to BOTH candidates "
      "below is a conservative (over-)estimate of baseline_cnn's share of the total.\n")

extrapolations = {}
for assumed_epochs, scenario in [
    (MAX_EPOCHS, "worst case: no early stop, full MAX_EPOCHS ceiling"),
    (9, "typical case: ASSUMED ~9 epochs to early-stop convergence -- an assumption, not measured"),
]:
    total_min = minutes_per_epoch * assumed_epochs * n_folds * n_candidates
    extrapolations[scenario] = total_min
    print(f"  {scenario}:")
    print(f"    {minutes_per_epoch:.2f} min/epoch x {assumed_epochs} epochs x {n_folds} folds x {n_candidates} candidates "
          f"= {total_min:.1f} min ({total_min / 60:.2f} hr)")

print("\nThis re-validation ran AFTER the Full Run section above (document order) -- the actual "
      "full run already used the confirmed BATCH_SIZE=32 + USE_TORCH_COMPILE=True from the "
      "Configuration cell, not any value computed in this section.")

=== Extrapolation using batch32 (1.38 min/epoch) ===
NOTE: this rate was measured on resnet50_unet only. baseline_cnn is a much smaller, from-scratch model (no decoder, single-channel input) and is expected to be substantially faster per epoch -- applying the resnet50_unet rate to BOTH candidates below is a conservative (over-)estimate of baseline_cnn's share of the total.

  worst case: no early stop, full MAX_EPOCHS ceiling:
    1.38 min/epoch x 15 epochs x 5 folds x 2 candidates = 206.6 min (3.44 hr)
  typical case: ASSUMED ~9 epochs to early-stop convergence -- an assumption, not measured:
    1.38 min/epoch x 9 epochs x 5 folds x 2 candidates = 124.0 min (2.07 hr)

This re-validation ran AFTER the Full Run section above (document order) -- the actual full run already used the confirmed BATCH_SIZE=32 + USE_TORCH_COMPILE=True from the Configuration cell, not any value computed in this section.
